# 05 — Test a New Song

Drop any MP3 or WAV path into the cell below.
Get back:
- Which cluster it belongs to
- Top-N most similar tracks from your library
- Where it sits on the UMAP map

**Requires**: notebook 04 to have been run (models/pipeline_phase2.pkl, models/index_phase2.*)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from anther_ml.embedding import load_mert, get_embedding
from anther_ml.cluster import load_pipeline, assign_cluster, cluster_summary
from anther_ml.similarity import SongIndex

print('imports ok')

## ⬇️ Change this cell to your song

In [ ]:
# ── DROP YOUR SONG PATH HERE ──────────────────────────────────────────────────
MY_SONG = '../data/audio/personal/my_song.mp3'
# ─────────────────────────────────────────────────────────────────────────────

TOP_K = 10   # how many nearest neighbors to return
PHASE = 2    # 1 = pre-computed features, 2 = MERT embeddings

## Load models

In [ ]:
pipeline_path = f'../models/pipeline_phase{PHASE}.pkl'
index_path    = f'../models/index_phase{PHASE}'
umap2d_path   = f'../models/embedding_2d_phase{PHASE}.npy'
labels_path   = f'../models/labels_phase{PHASE}.npy'

scaler, reducer, clusterer = load_pipeline(pipeline_path)
index = SongIndex.load(index_path)
embedding_2d = np.load(umap2d_path)
labels = np.load(labels_path)

print(f'Loaded Phase {PHASE} pipeline — {len(index.metadata)} tracks in index')

In [ ]:
if PHASE == 2:
    model, processor, device = load_mert()
    print(f'MERT on: {device}')
else:
    from anther_ml.features import extract_librosa_features
    model = processor = device = None

## Get embedding for the new song

In [ ]:
print(f'Processing: {Path(MY_SONG).name}')

if PHASE == 2:
    query_vec = get_embedding(model, processor, MY_SONG, device)
else:
    query_vec = extract_librosa_features(MY_SONG)

print(f'Embedding shape: {query_vec.shape}')

## Cluster assignment

In [ ]:
cluster_id, strength = assign_cluster(query_vec, scaler, reducer, clusterer)

print(f'Cluster:    {cluster_id}  (membership strength: {strength:.3f})')
if cluster_id == -1:
    print('  → Assigned to noise — this song is an outlier, unlike anything in the index.')
    print('  → Try lowering min_cluster_size in cluster.py or adding more similar songs.')
else:
    # What genres dominate this cluster?
    cluster_members = [m for m in index.metadata if m.get('cluster') == cluster_id]
    from collections import Counter
    genre_counts = Counter(m.get('genre', 'unknown') for m in cluster_members)
    print(f'  Cluster size: {len(cluster_members)} tracks')
    print(f'  Top genres:   {genre_counts.most_common(5)}')

## Nearest neighbors

In [ ]:
# Project query to same scaled space as the index
query_scaled = scaler.transform(query_vec.reshape(1, -1))[0]
results = index.query(query_scaled, top_k=TOP_K)

print(f'Top {TOP_K} most similar tracks to "{Path(MY_SONG).stem}":')
print(f'{"Rank":>4} | {"Score":>6} | {"Artist":<25} | {"Title":<35} | Genre')
print('-' * 95)
for r in results:
    artist = (r.get('artist') or '')[:24]
    name   = (r.get('name')   or '')[:34]
    genre  = r.get('genre', 'unknown') or 'unknown'
    print(f'{r["rank"]:>4} | {r["score"]:>6.4f} | {artist:<25} | {name:<35} | {genre}')

## Plot: where does this song land on the UMAP?

In [ ]:
# Project new song to 2D
q_scaled = scaler.transform(query_vec.reshape(1, -1))
song_2d = reducer.transform(q_scaled)  # (1, 2)

genre_labels = [m.get('genre', 'unknown') or 'unknown' for m in index.metadata]
unique_genres = sorted(set(genre_labels))
palette = sns.color_palette('tab20', n_colors=len(unique_genres))
g2c = {g: palette[i] for i, g in enumerate(unique_genres)}
colors = [g2c[g] for g in genre_labels]

fig, ax = plt.subplots(figsize=(13, 9))

# Background: all indexed songs
ax.scatter(embedding_2d[:, 0], embedding_2d[:, 1],
           c=colors, s=5, alpha=0.5, linewidths=0, zorder=1)

# Highlight top-10 neighbors
for r in results:
    idx = next((i for i, m in enumerate(index.metadata)
                if m.get('name') == r.get('name') and m.get('artist') == r.get('artist')), None)
    if idx is not None:
        ax.scatter(embedding_2d[idx, 0], embedding_2d[idx, 1],
                   c='orange', s=50, zorder=3, edgecolors='darkorange', linewidths=0.5)

# The new song
ax.scatter(song_2d[0, 0], song_2d[0, 1],
           c='red', s=150, zorder=5, edgecolors='black', linewidths=1.5,
           label=Path(MY_SONG).stem)
ax.annotate(Path(MY_SONG).stem,
            (song_2d[0, 0], song_2d[0, 1]),
            xytext=(8, 8), textcoords='offset points', fontsize=9, fontweight='bold')

handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=g2c[g], markersize=7, label=g)
           for g in unique_genres]
handles += [
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',
               markersize=8, label='Nearest neighbors'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
               markersize=10, label='Your song'),
]
ax.legend(handles=handles, title='Genre', bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=7)
ax.set_title(f'Phase {PHASE} — "{Path(MY_SONG).stem}" on the song map')
plt.tight_layout()
plt.savefig(f'../models/test_song_phase{PHASE}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to models/test_song_phase{PHASE}.png')